In [1]:
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [5]:
doc=documents_llm[0]

In [6]:
print(doc['id'])
print(doc['question'])
print(doc['answer'])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
len(user_prompt)

294

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['I found this course late — can I still start it and join in now?', 'Is it okay to enroll after the course has already begun?', 'If I’m joining the course now, can I still get the certificate?', 'What do I need to do to be eligible for the certificate if I join late?', 'Can I submit the final project after submissions close and still receive a certificate?']


In [15]:
print(result.questions)

['I found this course late — can I still start it and join in now?', 'Is it okay to enroll after the course has already begun?', 'If I’m joining the course now, can I still get the certificate?', 'What do I need to do to be eligible for the certificate if I join late?', 'Can I submit the final project after submissions close and still receive a certificate?']


In [16]:
'''
import urllib.request

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py"
urllib.request.urlretrieve(url, "evaluation_utils.py")
'''


'\nimport urllib.request\n\nurl = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py"\nurllib.request.urlretrieve(url, "evaluation_utils.py")\n'

In [17]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I just found out about it?', 'Is it too late to start the course now?', 'If I join late, can I still get a certificate?', 'What do I need to do to be eligible for the certificate?', 'Do I have to submit my project before submissions close to get the certificate?']


In [18]:
usage.input_tokens, usage.output_tokens

(207, 80)

In [37]:
usage

ResponseUsage(input_tokens=511, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=134, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=645)

In [19]:
# cost calculation

from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00036,
 'total_cost': 0.0005152500000000001}

In [20]:
# converting the result to a list of records with question and document id

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'}]

In [21]:
records

[{'question': 'Can I still join the course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'}]

In [22]:
import pandas as pd

In [23]:
df=pd.DataFrame(records)

In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'I just found this course late — am I still able to join it, or is it too late?',
   'document': '74eb249bbf'},
  {'question': 'Can I start the course now even though I missed the beginning?',
   'document': '74eb249bbf'},
  {'question': 'If I join after the course has already started, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': "What do I need to do to qualify for the certificate if I'm joining late?",
   'document': '74eb249bbf'},
  {'question': 'Is it okay to take the course now, and will I still be able to submit a project for certification?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=312))

In [27]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
ground_truth

[{'question': 'How exactly do I turn in the homework for this course?',
  'document': '0e38656cfb'},
 {'question': 'Do I need to host my homework code somewhere before submitting it, or can I just upload the files?',
  'document': '0e38656cfb'},
 {'question': 'Where can I find the homework assignments for my cohort, like the 2025 one?',
  'document': '0e38656cfb'},
 {'question': 'Is the homework submitted through GitHub or through the course platform form?',
  'document': '0e38656cfb'},
 {'question': 'Will I be able to see the homework solutions right after I submit, or only later?',
  'document': '0e38656cfb'},
 {'question': 'What changed in the 2025 version of the course compared to the older one?',
  'document': '226a4baf2f'},
 {'question': 'Is Flask still used in the deployment part, or did you switch to something else?',
  'document': '226a4baf2f'},
 {'question': 'Do the neural network lessons still use Keras, or are they now taught with PyTorch?',
  'document': '226a4baf2f'},
 {'

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents_llm, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

In [31]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

590

In [32]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.09157799999999998

In [33]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.09157799999999998

In [34]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [35]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [36]:
len(df_ground_truth)

590